# MNLP Homework 1 - Semantic Search  
Semantic search is an advanced search technique that focuses on understanding the user's intent and the contextual meaning of a query, rather than just matching exact keywords. It uses AI to find results that are conceptually related to what you asked, even if you use different words or synonyms.

In [1]:
# Pip needed installs
!pip install transformers
!pip install torch

In [2]:
# Import of dataset from huggingface
from datasets import load_dataset
mnlp_dataset = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Dataset is composed of 3 sets:  
- Train
- Test
- Blind
  
Only Train can be used to Train the model and Test to evaluate its performances.  
  
Available columns that can be used:  
- 'query',
- 'query_id',
- 'candidate_chunks',
- 'n_candidates',
- 'answer',
- 'answer_pos'

## Task [B1]  
Implement the two baseline semantic search systems:  
- distilbert-base-uncased
- all-MiniLM-L6-v2
  
For each model:  
1) Transform canditate_chunks into fixes-size dense vector representation (sentence embeddings) and memorize them
2) Encode query into fixes-size dense vector representation
3) Use cosine similarity metrics to find chunk closest to question
4) Use Hit@k metric to evaluate results

In [3]:
# Load train and test datasets
train_data = mnlp_dataset['train']
test_data = mnlp_dataset['test']

print(f"Train set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")
print(f"\nFirst training example:")
print(f"Query: {train_data[0]['query']}")
print(f"Number of candidates: {train_data[0]['n_candidates']}")
print(f"Answer: {train_data[0]['answer']}")
print(f"Answer position: {train_data[0]['answer_pos']}")

Train set size: 8000
Test set size: 2000

First training example:
Query: when did the smithsonian air and space museum open
Number of candidates: 45
Answer: The National Air and Space Museum of the Smithsonian Institution, also called the NASM, is a museum in Washington, D.C.. It was established in 1946 as the National Air Museum and opened its main building on the National Mall near L'Enfant Plaza in 1976. In 2016, the museum saw approximately 7.5 million visitors, making it the second most visited museum in the world, and the most visited museum in the United States.  The museum contains the Apollo 11 command module, the Friendship 7 capsule which was flown by John Glenn, Charles Lindbergh's Spirit of St. Louis, the Bell X-1 which broke the sound barrier, and the Wright brothers' plane near the entrance.
Answer position: 0


In [4]:
# Import required libraries for semantic search
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

# Install required packages if needed
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "torch", "-q"])

print(torch.cuda.is_available())  # Should be True for speedup
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

False
CPU


In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


def evaluate_semantic_search(model_name, dataset, split_name="test"):
    """
    Evaluate semantic search system using a given HuggingFace transformer model.
    Uses mean pooling to create sentence embeddings.
    
    For each query in dataset:
    1) Transform candidate_chunks into fixed-size dense vector representation (sentence embeddings) and memorize them
    2) Encode query into fixed-size dense vector representation
    3) Use cosine similarity metrics to find chunk closest to question
    4) Use Hit@k metric to evaluate results
    """
    print(f"\n{'='*60}")
    print(f"Evaluating model: {model_name}")
    print(f"Dataset: {split_name} ({len(dataset)} samples)")
    print(f"{'='*60}\n")
    
    # Load model and tokenizer
    print(f"[1/4] Loading model {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    print(f"✓ Model ready on {device}\n")
    
    def encode_texts(texts):
        """Encode texts into embeddings using mean pooling"""
        encoded = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=256)
        encoded = {k: v.to(device) for k, v in encoded.items()}
        
        with torch.no_grad():
            model_output = model(**encoded)
        
        sentence_embeddings = mean_pooling(model_output, encoded['attention_mask'])
        return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1).cpu().numpy()
    
    # Metrics to track
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0
    total_samples = len(dataset)
    
    print(f"[2/4] Processing {total_samples} queries...")
    print(f"Progress | Hit@1 | Hit@3 | Hit@5")
    print("-" * 35)
    
    # Process each query
    for idx, example in enumerate(tqdm(dataset, desc="Processing queries")):
        query = example['query']
        candidate_chunks = example['candidate_chunks']
        answer_pos = example['answer_pos']
        
        # Step 2: Encode query into fixed-size dense vector
        query_embedding = encode_texts([query])[0]
        
        # Step 1: Transform candidate chunks into fixed-size dense vectors and memorize them
        chunk_embeddings = encode_texts(candidate_chunks)
        
        # Step 3: Use cosine similarity to find the closest chunk
        similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
        
        # Get the ranking of chunks sorted by similarity (highest first)
        ranked_indices = np.argsort(similarities)[::-1]
        
        # Step 4: Compute Hit@k metrics
        # Find position of the correct answer in the ranking
        for rank, idx_pos in enumerate(ranked_indices):
            if idx_pos == answer_pos:
                if rank < 1:
                    hits_at_1 += 1
                if rank < 3:
                    hits_at_3 += 1
                if rank < 5:
                    hits_at_5 += 1
                break
        
        # Print progress every 10% of dataset
        if (idx + 1) % max(1, total_samples // 10) == 0:
            progress_pct = (idx + 1) / total_samples * 100
            h1 = (hits_at_1 / (idx + 1))
            h3 = (hits_at_3 / (idx + 1))
            h5 = (hits_at_5 / (idx + 1))
            print(f"{progress_pct:5.1f}%  | {h1:.3f} | {h3:.3f} | {h5:.3f}")
    
    # Calculate final metrics
    hit_at_1 = hits_at_1 / total_samples
    hit_at_3 = hits_at_3 / total_samples
    hit_at_5 = hits_at_5 / total_samples
    
    print(f"100.0%  | {hit_at_1:.3f} | {hit_at_3:.3f} | {hit_at_5:.3f}")
    print(f"\n[3/4] Evaluation complete!")
    print(f"[4/4] Results ready\n")
    
    return {
        'model': model_name,
        'hit@1': hit_at_1,
        'hit@3': hit_at_3,
        'hit@5': hit_at_5,
    }

In [ ]:
# Baseline 1: distilbert-base-uncased
model_1_results = evaluate_semantic_search('distilbert-base-uncased', test_data, 'test')

print(f"\n{'='*60}")
print(f"Results for {model_1_results['model']}")
print(f"{'='*60}")
print(f"Hit@1:  {model_1_results['hit@1']:.4f}")
print(f"Hit@3:  {model_1_results['hit@3']:.4f}")
print(f"Hit@5:  {model_1_results['hit@5']:.4f}")

Encoding chunks:   7%|▋         | 110/1575 [50:56<11:20:36, 27.87s/it]

: 

In [ ]:
# Baseline 2: all-MiniLM-L6-v2
model_2_results = evaluate_semantic_search('sentence-transformers/all-MiniLM-L6-v2', test_data, 'test')

print(f"\n{'='*60}")
print(f"Results for {model_2_results['model']}")
print(f"{'='*60}")
print(f"Hit@1:  {model_2_results['hit@1']:.4f}")
print(f"Hit@3:  {model_2_results['hit@3']:.4f}")
print(f"Hit@5:  {model_2_results['hit@5']:.4f}")

In [ ]:
# Comparison of both baseline models
import pandas as pd

results_df = pd.DataFrame([model_1_results, model_2_results])
print(f"\n{'='*80}")
print("BASELINE MODELS COMPARISON")
print(f"{'='*80}")
print(results_df.to_string(index=False))
print(f"{'='*80}\n")

# Determine which model performed better
if model_1_results['hit@1'] > model_2_results['hit@1']:
    print(f"✓ {model_1_results['model']} performed better overall")
else:
    print(f"✓ {model_2_results['model']} performed better overall")